# 08 · Observability traces — what Langfuse sees

The trace contract, taught offline; a marked opt-in cell for the live path.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- node -> span-name map (verb-first, stable)
- the deterministic trace identity (seeded from filename)
- score + prompt-link contracts

**Honesty label:** OFFLINE — shapes computed from the pipeline's own seeding functions and fixture contracts, NOT a live Langfuse fetch.

## Node -> span map

In [2]:
for node, span in lab.NODE_SPAN_NAMES.items():
    print(f"{node:16s} -> {span}")


intake -> intake-document
classify         -> classify-document
retry_classify   -> classify-document
review_classify  -> classify-document
extract          -> extract-fields
retry_extract    -> extract-fields
judge_verify     -> judge-verify
arbiter          -> arbitrate-verdict
human_review     -> route-for-review
boss_escalation  -> adjudicate-conflict
compile_report   -> compile-report
catalog_write    -> write-catalog
archive          -> archive-document


## Trace identity (computed, not fetched)

In [3]:
c = lab.trace_contract(filename="papertrail.txt", matter_id="LAB-MATTER")
print("trace_id:  ", c["trace_id"])
print("session_id:", c["session_id"])
print("name:      ", c["name"])
print("spans:     ", len(c["span_names"]), "node spans, verb-first")
print("scores:    ", c["score_names"])


2026-08-24 18:17:20 [debug    ] langfuse_not_configured_no_secret_key


2026-08-24 18:17:21 [debug    ] score_configs_validated        count=37 registry=llm-dojo-scoring


2026-08-24 18:17:21 [info     ] checkpointer_initialized       backend=memory


trace_id:   (disabled client)
session_id: LAB-MATTER
name:       document-pipeline
spans:      13 node spans, verb-first
scores:     ['completeness']


### The contract in prose

- trace name `document-pipeline`; `session_id = matter_id`
- generations carry model name + token usage (auto-traced via
  `langfuse.openai`)
- curated inputs (file metadata, not raw payloads) — PII stays local
- `langfuse_prompt=` links each generation to its managed prompt
- environment tags (`mock` here, `live` in prod) on every trace

Notebook 06 showed the same run's `trace_id` stamped on manifest, catalog
row, and audit chain — that is the join key between the four local
surfaces and this fifth, remote one.

## OPTIONAL live cell (network + keys required)

<!-- NB-OPT-IN-NETWORK: guarded live-trace cell; skipped unless keys are set -->

In [4]:
# OPT-IN: runs one traced mock-LLM document against real Langfuse.
import os

if os.environ.get("LANGFUSE_SECRET_KEY"):
    with lab.lab_sandbox() as env:
        os.environ["OBSERVABILITY_PROVIDER"] = "langfuse"
        r = lab.run_document(env, lab.DOC_CONTRACT,
                             classification=lab.CLASSIFY_CONTRACT_HIGH,
                             extraction=lab.EXTRACT_HIGH,
                             filename="traced_live.txt")
        print("ran; fetch trace", r["final"].get("trace_id"),
              "from your Langfuse dashboard")
else:
    print("skipped: LANGFUSE_SECRET_KEY not set (this is the default)")


skipped: LANGFUSE_SECRET_KEY not set (this is the default)


## Where to go next

- back to **00 · pipeline_anatomy** — the static map
- `scripts/sync_dashboards.py` — the health dashboards this data feeds